# STRAP on MedIAnomaly: STRAP as a decision layer on an existing autoencoder

**Goal.** Apply STRAP on top of the reconstruction AE of the MedIAnomaly benchmark (Cai et al., MICCAI 2024; official code `caiyu6666/MedIAnomaly`) and test whether it keeps that AE's advantage over more complex reconstruction models while producing detection decisions without threshold selection.

**Base autoencoders** (same architecture, data, preprocessing and training schedule as the official code):
- `dstar`: AE with the per-dataset optimal latent size d* (RSNA 4, VinDr-CXR 4, Brain Tumor 32, BraTS2021 16), i.e. AE-d*.
- `d16`: AE with the default latent size 16 (identical to `dstar` on BraTS2021, so it is run once there).

**Regularization path:** λ ∈ {0, 1e-4, 1e-3, 1e-2, 1e-1} applied as λ·mean|z| on the latent code. λ=0 is the unmodified base AE.

**Seeds:** 0, 1, 2 (the official code uses 3 repeats on a fixed train/test split).

**Calibration set.** For each seed, a random 20% of the (normal) training images is held out and never seen by the autoencoders. Their reconstruction errors are the reference distribution for the decision test, so each per-λ test has false positive rate α on unseen normal images. The autoencoders train on the remaining 80%.

**Decision test.** Per λ, Box-Cox is fitted on the calibration errors, and each test error gets the one-sided prediction-interval statistic t = (y − ȳ) / (s·√(1+1/n)), compared with Student-t(n−1). Rules over the path: λ=0 only, union (≥1), ≥2, majority (≥3), all 5, Bonferroni union (α/5). Ranking uses the maximum t over the path.

**No selection on test data.** Nothing is tuned on the test set. The analysis reports STRAP at λ=0 and STRAP aggregated over the whole path; the oracle threshold appears only as a labelled upper bound.

**Saved per run:** image-level reconstruction errors for the training images, the calibration images and the test images, plus test labels. Everything else (test statistics, decision rules, false positive rates, label-free threshold baselines, normality tests) is computed in the analysis cell from these files, so new analyses never require retraining.

**Runtime.** 4 datasets × 7 base configurations × 5 λ × 3 seeds = 105 training runs. Runs are skipped if their output file already exists, so the notebook can resume after a disconnect. Set `SEEDS = [0]` for a first pass.

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────
!pip install -q SimpleITK
import os, sys, time, json, random, glob, subprocess
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
from scipy import stats
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_recall_curve

if not os.path.exists('/content/MedIAnomaly'):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/caiyu6666/MedIAnomaly.git', '/content/MedIAnomaly'], check=True)
sys.path.insert(0, '/content/MedIAnomaly/reconstruction')
from networks.ae import AE                          # official architecture
from dataloaders.dataload import MedAD, BraTSAD     # official data loaders and splits

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE, torch.cuda.get_device_name(0) if DEVICE == 'cuda' else '')

In [ ]:
# ── Google Drive (outputs and optional data cache) ─────────────────────
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/STRAP_MedIAnomaly'
RUNS_DIR   = os.path.join(DRIVE_ROOT, 'runs_cal')      # one .npz per (dataset, base, seed, lambda)
TAR_CACHE  = os.path.join(DRIVE_ROOT, 'data_tar')      # cached Zenodo archives
os.makedirs(RUNS_DIR, exist_ok=True); os.makedirs(TAR_CACHE, exist_ok=True)
print('Outputs:', DRIVE_ROOT)

In [ ]:
# ── Data: MedIAnomaly pre-processed datasets (Zenodo record 12677223) ───
DATA_ROOT = os.path.expanduser('~/MedIAnomaly-Data')     # path expected by the official loaders
ARCHIVES  = {'rsna': ('RSNA.tar.gz', 'RSNA'), 'vin': ('VinCXR.tar.gz', 'VinCXR'),
             'brain': ('BrainTumor.tar.gz', 'BrainTumor'), 'brats': ('BraTS2021.tar.gz', 'BraTS2021')}
ZENODO = 'https://zenodo.org/records/12677223/files/{}?download=1'
os.makedirs(DATA_ROOT, exist_ok=True)

for ds, (tar, folder) in ARCHIVES.items():
    if os.path.isdir(os.path.join(DATA_ROOT, folder)):
        print(f'{ds}: ready'); continue
    cached = os.path.join(TAR_CACHE, tar)
    if not os.path.exists(cached):
        print(f'{ds}: downloading {tar} ...')
        subprocess.run(['wget', '-q', '-O', cached, ZENODO.format(tar)], check=True)
    print(f'{ds}: extracting ...')
    subprocess.run(['tar', '-xzf', cached, '-C', DATA_ROOT], check=True)
    assert os.path.isdir(os.path.join(DATA_ROOT, folder)), f'unexpected archive layout for {tar}'
print(os.listdir(DATA_ROOT))

In [ ]:
# ── Configuration (official MedIAnomaly settings unless noted) ─────────
DATASETS   = ['rsna', 'vin', 'brain', 'brats']
D_STAR     = {'rsna': 4, 'vin': 4, 'brain': 32, 'brats': 16}      # Cai et al. (2024), AE-d*
BASES      = ['dstar', 'd16']                                     # run order = priority
LAMBDAS    = [0.0, 1e-4, 1e-3, 1e-2, 1e-1]
SEEDS      = [0, 1, 2]
EPOCHS     = {'rsna': 250, 'vin': 250, 'brain': 600, 'brats': 250}
BATCH, LR, INPUT_SIZE, HIDDEN = 64, 1e-3, 64, 1024
ALPHAS     = [0.05, 0.10]
CAL_FRAC   = 0.20                                         # share of training images held out for calibration

def latent_size(ds, base): return D_STAR[ds] if base == 'dstar' else 16
def lam_tag(l): return 'lam0' if l == 0 else f'lam1e{abs(int(round(np.log10(l))))}'
def run_file(ds, d, seed, lam): return os.path.join(RUNS_DIR, f'{ds}_d{d}_seed{seed}_{lam_tag(lam)}.npz')

# configurations keyed by latent size, so BraTS2021 d*=16 and d16 share one run
CONFIGS = []
for ds in DATASETS:
    for d in dict.fromkeys(latent_size(ds, b) for b in BASES):
        CONFIGS.append((ds, d))
print(len(CONFIGS), 'configurations x', len(LAMBDAS), 'lambdas x', len(SEEDS), 'seeds =',
      len(CONFIGS) * len(LAMBDAS) * len(SEEDS), 'runs')

In [ ]:
# ── Data loading, training and reconstruction errors ───────────────────
TRANSFORM = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
_DATA = {}
def get_sets(ds):
    if ds not in _DATA:
        path = os.path.join(DATA_ROOT, ARCHIVES[ds][1])
        cls = BraTSAD if ds == 'brats' else MedAD
        _DATA[ds] = (cls(main_path=path, img_size=INPUT_SIZE, transform=TRANSFORM, mode='train'),
                     cls(main_path=path, img_size=INPUT_SIZE, transform=TRANSFORM, mode='test'))
    return _DATA[ds]

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def build_ae(d):
    return AE(input_size=INPUT_SIZE, in_planes=1, base_width=16, expansion=1, mid_num=HIDDEN,
              latent_size=d, en_num_layers=1, de_num_layers=1).to(DEVICE)

def train(net, train_set, lam, epochs):
    # drop a final batch of size 1, which BatchNorm1d cannot train on
    loader = DataLoader(train_set, batch_size=BATCH, shuffle=True,
                        drop_last=(len(train_set) % BATCH == 1))
    opt = torch.optim.Adam(net.parameters(), lr=LR, weight_decay=0)
    net.train()
    for _ in range(epochs):
        for batch in loader:
            x = batch['img'].to(DEVICE)
            out = net(x)
            loss = ((x - out['x_hat']) ** 2).mean() + lam * out['z'].abs().mean()
            opt.zero_grad(); loss.backward(); opt.step()
    return net

@torch.no_grad()
def image_errors(net, dataset):
    net.eval(); errs, labels = [], []
    for batch in DataLoader(dataset, batch_size=256, shuffle=False):
        x = batch['img'].to(DEVICE)
        errs.append(((x - net(x)['x_hat']) ** 2).mean(dim=[1, 2, 3]).cpu().numpy())
        labels.append(np.asarray(batch['label']))
    return np.concatenate(errs), np.concatenate(labels)
print('Helpers ready.')

In [ ]:
# ── Main loop (resumable) ──────────────────────────────────────────────
from torch.utils.data import Subset
todo = [(ds, d, s, l) for s in SEEDS for (ds, d) in CONFIGS for l in LAMBDAS
        if not os.path.exists(run_file(ds, d, s, l))]
print(f'{len(todo)} runs to do')
for i, (ds, d, seed, lam) in enumerate(todo):
    t0 = time.time()
    train_set, test_set = get_sets(ds)
    # calibration split: fixed per seed, identical for every lambda and base model
    perm = np.random.default_rng(10_000 + seed).permutation(len(train_set))
    n_cal = int(round(CAL_FRAC * len(train_set)))
    fit_set, cal_set = Subset(train_set, perm[n_cal:]), Subset(train_set, perm[:n_cal])
    set_seed(seed)
    net = train(build_ae(d), fit_set, lam, EPOCHS[ds])
    tr_err, _ = image_errors(net, fit_set)
    cal_err, _ = image_errors(net, cal_set)
    te_err, y = image_errors(net, test_set)
    np.savez_compressed(run_file(ds, d, seed, lam), train_err=tr_err.astype(np.float32),
                        cal_err=cal_err.astype(np.float32), test_err=te_err.astype(np.float32),
                        y=y.astype(np.int8))
    auc = roc_auc_score(y, te_err)
    print(f'[{i+1}/{len(todo)}] {ds} d={d} seed={seed} {lam_tag(lam)}: '
          f'raw-error AUROC={auc:.4f}  ({time.time()-t0:.0f}s)')
    del net; torch.cuda.empty_cache()
print('Done.')

In [ ]:
# ── Analysis: all metrics from the saved errors ────────────────────────
from scipy.special import boxcox as boxcox_transform

def pi_stat(ref, e):
    # Box-Cox prediction-interval statistic; Box-Cox fitted on the reference errors
    rb, lmb = stats.boxcox(np.maximum(ref, 1e-12)); eb = boxcox_transform(np.maximum(e, 1e-12), lmb)
    n = len(rb)
    return (eb - rb.mean()) / (rb.std(ddof=1) * np.sqrt(1 + 1 / n)), n - 1

def prf(y, det):
    tp = int((det & (y == 1)).sum()); fp = int((det & (y == 0)).sum())
    p = tp / max(1, int(det.sum())); r = tp / max(1, int(y.sum()))
    return {'f1': 0.0 if p + r == 0 else 2 * p * r / (p + r), 'precision': p, 'recall': r,
            'fpr': fp / max(1, int((y == 0).sum()))}

def oracle_best_f1(y, s):
    p, r, _ = precision_recall_curve(y, s); return float(np.max(2 * p * r / np.maximum(p + r, 1e-12)))

def normality(x, rng):
    x = x if len(x) <= 5000 else rng.choice(x, 5000, replace=False)
    xb, _ = stats.boxcox(np.maximum(x, 1e-12))
    return {'sw_p_raw': stats.shapiro(x).pvalue, 'ad_raw': stats.anderson(x).statistic,
            'sw_p_boxcox': stats.shapiro(xb).pvalue, 'ad_boxcox': stats.anderson(xb).statistic}

rows, norm_rows = [], []
rng = np.random.default_rng(0)
for (ds, d) in CONFIGS:
    for seed in SEEDS:
        files = [run_file(ds, d, seed, l) for l in LAMBDAS]
        if not all(os.path.exists(f) for f in files): continue
        R = [np.load(f) for f in files]; y = R[0]['y'].astype(int)
        tr = [r['train_err'].astype(np.float64) for r in R]
        ca = [r['cal_err'].astype(np.float64) for r in R]
        te = [r['test_err'].astype(np.float64) for r in R]
        K = len(LAMBDAS)
        m = {'dataset': ds, 'd': d, 'seed': seed, 'n_cal': len(ca[0])}
        # base AE ranking (lambda = 0, raw error); reproduction check against Cai et al. (trained on 80%)
        m['auroc_raw_lam0'] = roc_auc_score(y, te[0]); m['ap_raw_lam0'] = average_precision_score(y, te[0])
        # prediction-interval test with calibration errors (main) and with training errors (comparison)
        for ref_name, REF in [('cal', ca), ('train', tr)]:
            TT, DF = zip(*[pi_stat(REF[k], te[k]) for k in range(K)])
            TT = np.stack(TT); P = np.stack([stats.t.sf(TT[k], DF[k]) for k in range(K)])
            agg = TT.max(axis=0)
            m[f'auroc_{ref_name}_agg'] = roc_auc_score(y, agg); m[f'ap_{ref_name}_agg'] = average_precision_score(y, agg)
            for a in ALPHAS:
                D = P < a; votes = D.sum(axis=0)
                rules = {'lam0': D[0], 'union': votes >= 1, 'ge2': votes >= 2, 'majority': votes >= 3,
                         'all': votes == K, 'bonferroni': (P < a / K).any(axis=0)}
                for r, det in rules.items():
                    for k2, v in prf(y, det).items(): m[f'{k2}_{ref_name}_{r}_a{a}'] = v
                for k in range(K):
                    m[f'fpr_{ref_name}_{lam_tag(LAMBDAS[k])}_a{a}'] = float(D[k][y == 0].mean())
        # label-free thresholds on the base AE (lambda = 0) errors
        for name, thr in [('cal_q95', np.quantile(ca[0], 0.95)), ('train_p95', np.quantile(tr[0], 0.95)),
                          ('train_p99', np.quantile(tr[0], 0.99)), ('train_mean3sd', tr[0].mean() + 3 * tr[0].std(ddof=1))]:
            for k2, v in prf(y, te[0] > thr).items(): m[f'{k2}_thr_{name}'] = v
        m['f1_oracle_lam0'] = oracle_best_f1(y, te[0])          # uses test labels: upper bound only
        for b in [b for b in BASES if latent_size(ds, b) == d]: rows.append({**m, 'base': b})
        for k, l in enumerate(LAMBDAS):
            norm_rows.append({'dataset': ds, 'd': d, 'seed': seed, 'lambda': l, 'reference': 'calibration', **normality(ca[k], rng)})

per_seed = pd.DataFrame(rows); norm = pd.DataFrame(norm_rows)
summary = per_seed.drop(columns='seed').groupby(['dataset', 'base', 'd']).agg(['mean', 'std'])
summary.columns = [f'{a}_{b}' for a, b in summary.columns]
per_seed.to_csv(os.path.join(DRIVE_ROOT, 'medianomaly_per_seed.csv'), index=False)
summary.reset_index().to_csv(os.path.join(DRIVE_ROOT, 'medianomaly_summary.csv'), index=False)
norm.to_csv(os.path.join(DRIVE_ROOT, 'medianomaly_normality.csv'), index=False)

show = ['auroc_raw_lam0', 'auroc_cal_agg', 'ap_raw_lam0', 'ap_cal_agg', 'fpr_cal_lam0_a0.05', 'fpr_cal_union_a0.05',
        'fpr_cal_bonferroni_a0.05', 'f1_cal_majority_a0.05', 'f1_cal_union_a0.05', 'f1_thr_cal_q95', 'f1_oracle_lam0']
print((100 * summary[[f'{c}_mean' for c in show]]).round(1).to_string())
print()
print('Saved: medianomaly_per_seed.csv, medianomaly_summary.csv, medianomaly_normality.csv')